## Goal

The goal of this notebook is to find the occurrences of place names for each reign in `regnal_files`.  
This process will be expanded to all reigns in [`1577_working.xml`](../1577_files/1577_working.xml).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Old source of truth
# df_holinshed_names_unique = pd.read_csv("../name_extraction/entity_names/holinshed_names_unique.csv")
# place_names = df_holinshed_names_unique.loc[
#     df_holinshed_names_unique["type"].eq("place")
# ].copy()

# Select place names based on source of truth
place_names = pd.read_csv("../name_extraction/entity_names/holinshed_places.csv")
place_names.info()
print(f"Number of unique places: {len(place_names)}")

In [ ]:
# Define confidence categories
confidence_bins = [0, 0.25, 0.50, 0.75, 1.00]
confidence_labels = ["low", "medium", "high", "very high"]
print(f"Low confidence bin: [{confidence_bins[0]},{confidence_bins[1]})", flush=True)
print(f"Medium confidence bin: [{confidence_bins[1]},{confidence_bins[2]})", flush=True)
print(f"High confidence bin: [{confidence_bins[2]},{confidence_bins[3]})", flush=True)
print(f"Very high confidence bin: [{confidence_bins[3]},{confidence_bins[4]})", flush=True)

# Categorize places in confidence categories based on pre-defined confidence bins
place_names["confidence_category"] = pd.cut(
    place_names["confidence"],
    bins=confidence_bins,
    labels=confidence_labels,
    include_lowest=True,
    right=False
)

# Count names in each confidence category
confidence_counts = (
    place_names["confidence_category"]
    .value_counts()
    .sort_index()
    .rename_axis("confidence_level")
    .reset_index(name="count")
)

display(confidence_counts)


veryhigh_conf_places = place_names.loc[place_names["confidence"].ge(confidence_bins[3])].copy()
print(
    f"We will work with the {len(veryhigh_conf_places)} very high confidence places, or {(len(veryhigh_conf_places) / len(place_names))*100:.2f}% of all places."
)

In [ ]:
import re
from pathlib import Path
import xml.etree.ElementTree as ET

# Find the repository root whether the notebook runs from notebooks/ or the repo root.
repo_root = Path.cwd().resolve()
if not (repo_root / "1577_files").is_dir():
    repo_root = repo_root.parent

xml_dir = repo_root / "1577_files" / "regnal_files"
xml_files = sorted(xml_dir.glob("*.xml"))
if not xml_files:
    raise FileNotFoundError(f"No regnal XML files found in {xml_dir}")

places = veryhigh_conf_places.copy()

print(
    f"Searching for {len(places)} high-confidence places in {len(xml_files)} regnal files."
)


all_results = []
# Iterate over each XML file and search for high-confidence place names
for xml_file in xml_files:
    print(f"Processing {xml_file.name}...")
    root = ET.parse(xml_file).getroot()

    # Keep adjacent XML text nodes from being concatenated into a single word.
    xml_text = re.sub(r"\s+", " ", " ".join(root.itertext()))
    word_count = len(xml_text.split())

    # Iterate over each high-confidence place and count occurrences in the XML text
    for _, row in places.iterrows():
        # Get actual name and any spelling variants for searching
        search_keys = [str(row["name"])]
        if pd.notna(row["spelling_variants"]):
            search_keys.extend(str(row["spelling_variants"]).split("|"))
        search_keys = {key.strip() for key in search_keys if key.strip()}

        total_count = sum(
            # Count occurrences of the current search key in the XML text
            len(
                re.findall(
                    rf"(?<!\w){re.escape(key)}(?!\w)", xml_text, flags=re.IGNORECASE
                )
            )
            for key in search_keys
        )

        all_results.append(
            {
                "file": xml_file.stem,
                "place_name": row["name"],
                "xml_count": total_count,
                "word_count": word_count,
            }
        )

print("Finished processing.")
results = pd.DataFrame(all_results)

# Build final DataFrame of place counts by reign
place_counts_by_reign = results.pivot_table(
    index="place_name", columns="file", values="xml_count", aggfunc="sum", fill_value=0
).astype(int)

place_counts_by_reign.info()


In [ ]:
# 5 most mentioned places
print("Top 5 most mentioned places:")
display(place_counts_by_reign.loc[
    place_counts_by_reign.sum(axis=1).sort_values(ascending=False).index
].head(5))


# Number of distinct places with at least one match in each XML file
places_with_matches = (place_counts_by_reign > 0).sum()

# Total number of place mentions in each XML file
total_mentions = place_counts_by_reign.sum()

coverage = pd.DataFrame({
    "word_count": results.groupby("file")["word_count"].first(),
    "places_found": places_with_matches,
    "total_mentions": total_mentions
})

print("Coverage:")
display(coverage)


# Create a heatmap
top_20_places = (
    place_counts_by_reign.sum(axis=1)
    .sort_values(ascending=False)
    .head(20)
    .index
)

top_20_by_reign = place_counts_by_reign.loc[top_20_places]

plt.figure(figsize=(12, 8))

sns.heatmap(
    top_20_by_reign,
    cmap="Greens",
    linewidths=0.5
)

plt.title("Top 20 Places Across Regnal Files")
plt.xlabel("Regnal file")
plt.ylabel("Place")
plt.tight_layout()
plt.show()

# TODO: Output heatmap based on relative sizes of reigns